In [56]:
# Install/import required libraries.
# In Kaggle, the packages are usually already available.
!pip -q install nltk scikit-learn pandas numpy

import re
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
import nltk

from nltk.corpus import brown, gutenberg
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.tag import hmm
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

# Download all resources required by this notebook.
resources = [
    ("corpora", "brown"),
    ("corpora", "gutenberg"),
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
]

for category, resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as error:
        print(f"Could not download {resource}: {error}")

print("Libraries and corpora are ready.")


Libraries and corpora are ready.


## Question 1 — Brown Corpus and N-gram Statistics



In [57]:
# Q1(a-b): Load the Brown news category and preprocess at least 5,000 tokens.
brown_tokens = [
    token.lower()
    for token in brown.words(categories="news")
    if token.isalpha()
]

tokens_q1 = brown_tokens[:5000]

print("Number of selected tokens:", len(tokens_q1))
print("First 30 preprocessed tokens:")
print(tokens_q1[:30])


Number of selected tokens: 5000
First 30 preprocessed tokens:
['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place', 'the', 'jury', 'further', 'said', 'in', 'presentments', 'that', 'the', 'city']


In [58]:
# Q1(c): Calculate total token count and vocabulary size.
total_tokens = len(tokens_q1)
vocabulary = set(tokens_q1)
vocabulary_size = len(vocabulary)

q1_statistics = pd.DataFrame({
    "Measure": ["Total tokens", "Vocabulary size"],
    "Value": [total_tokens, vocabulary_size]
})

display(q1_statistics)


,Measure,Value
0,Total tokens,5000
1,Vocabulary size,1511


In [59]:
# Q1(d): Generate unigram, bigram, and trigram sequences.
unigrams_q1 = list(ngrams(tokens_q1, 1))
bigrams_q1 = list(ngrams(tokens_q1, 2))
trigrams_q1 = list(ngrams(tokens_q1, 3))

print("Number of unigram sequences:", len(unigrams_q1))
print("Number of bigram sequences:", len(bigrams_q1))
print("Number of trigram sequences:", len(trigrams_q1))


Number of unigram sequences: 5000
Number of bigram sequences: 4999
Number of trigram sequences: 4998


In [60]:
# Q1(e): Display the ten most frequent examples of each n-gram.
unigram_counts_q1 = Counter(unigrams_q1)
bigram_counts_q1 = Counter(bigrams_q1)
trigram_counts_q1 = Counter(trigrams_q1)

def frequency_table(counter, column_name):
    rows = [
        (" ".join(gram), count)
        for gram, count in counter.most_common(10)
    ]
    return pd.DataFrame(rows, columns=[column_name, "Frequency"])

print("Top 10 unigrams")
display(frequency_table(unigram_counts_q1, "Unigram"))

print("Top 10 bigrams")
display(frequency_table(bigram_counts_q1, "Bigram"))

print("Top 10 trigrams")
display(frequency_table(trigram_counts_q1, "Trigram"))


Top 10 unigrams


,Unigram,Frequency
0,the,388
1,of,204
2,to,149
3,a,129
4,in,102
5,and,97
6,for,63
7,that,51
8,would,50
9,said,46


Top 10 bigrams


,Bigram,Frequency
0,of the,64
1,in the,42
2,on the,16
3,the state,15
4,the jury,14
5,for the,14
6,would be,14
7,to the,13
8,that the,11
9,at the,10


Top 10 trigrams


,Trigram,Frequency
0,the jury said,7
1,of the ward,6
2,the grand jury,4
3,is expected to,4
4,some of the,4
5,the precinct of,4
6,precinct of the,4
7,up to days,4
8,the fulton county,3
9,jury said it,3


### Q1(f) Interpretation

A language corpus provides real examples of how words occur in context. Unigram frequencies show individual word usage, while bigram and trigram frequencies show local sequential patterns. These statistics are useful for language modeling, autocomplete, spelling correction, speech recognition, and text generation.


## Question 2 — Bigram Language Model with Add-one Smoothing




In [61]:
# Q2(a-b): Load Gutenberg text, tokenize it, and normalize it.
raw_emma = gutenberg.raw("austen-emma.txt")

emma_tokens = [
    token.lower()
    for token in word_tokenize(raw_emma)
    if token.isalpha()
]

# Use a manageable but sufficiently large portion for classroom execution.
emma_tokens = emma_tokens[:20000]

print("Number of tokens used:", len(emma_tokens))
print("First 30 tokens:", emma_tokens[:30])


Number of tokens used: 20000
First 30 tokens: ['emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of']


In [62]:
# Q2(c): Construct unigram and bigram frequency counts.
unigram_counts_q2 = Counter(emma_tokens)
bigram_counts_q2 = Counter(zip(emma_tokens[:-1], emma_tokens[1:]))

V = len(unigram_counts_q2)

print("Vocabulary size:", V)
print("Number of unique bigrams:", len(bigram_counts_q2))

print("\nTop 10 unigrams:")
display(frequency_table(unigram_counts_q2, "Unigram"))

print("Top 10 bigrams:")
display(frequency_table(bigram_counts_q2, "Bigram"))


Vocabulary size: 2544
Number of unique bigrams: 12672

Top 10 unigrams:


,Unigram,Frequency
0,a n d,657
1,t o,630
2,o f,604
3,t h e,586
4,a,530
5,i,380
6,h e r,349
7,s h e,295
8,n o t,295
9,i n,289


Top 10 bigrams:


,Bigram,Frequency
0,of the,77
1,to be,77
2,of her,57
3,it was,52
4,in the,51
5,a very,51
6,she had,50
7,she was,45
8,to have,39
9,i am,38


In [63]:
# Q2(d): Implement add-one smoothing.
def add_one_bigram_probability(previous_word, current_word):
    bigram_count = bigram_counts_q2[(previous_word, current_word)]
    previous_count = unigram_counts_q2[previous_word]
    return (bigram_count + 1) / (previous_count + V)

def sentence_probability(sentence):
    words = [
        token.lower()
        for token in word_tokenize(sentence)
        if token.isalpha()
    ]

    if len(words) < 2:
        return 1.0

    log_probability = 0.0

    for previous_word, current_word in zip(words[:-1], words[1:]):
        probability = add_one_bigram_probability(
            previous_word, current_word
        )
        log_probability += math.log(probability)

    return math.exp(log_probability)

def sentence_log_probability(sentence):
    words = [
        token.lower()
        for token in word_tokenize(sentence)
        if token.isalpha()
    ]

    if len(words) < 2:
        return 0.0

    return sum(
        math.log(add_one_bigram_probability(previous_word, current_word))
        for previous_word, current_word in zip(words[:-1], words[1:])
    )

print("Smoothing function created.")


Smoothing function created.


In [64]:
# Q2(e): Calculate scores for at least three test sentences.
test_sentences_q2 = [
    "She was a very kind woman.",
    "The house was large and beautiful.",
    "The spacecraft moved around the planet."
]

q2_results = pd.DataFrame({
    "Test sentence": test_sentences_q2,
    "Log probability": [
        sentence_log_probability(sentence)
        for sentence in test_sentences_q2
    ],
    "Probability": [
        sentence_probability(sentence)
        for sentence in test_sentences_q2
    ]
})

display(q2_results)


,Test sentence,Log probability,Probability
0,She was a very kind woman.,-27.151494,1.615311e-12
1,The house was large and beautiful.,-37.455615,5.410447e-17
2,The spacecraft moved around the planet.,-39.622448,6.197112e-18


## Question 4 — Ambiguity, Coreference, and Domain-specific Language


In [70]:
# Q4(a-b): Create at least five ambiguity examples and identify intended meanings.
ambiguity_examples = pd.DataFrame({
    "Sentence": [
        "I went to the bank to deposit money.",
        "The bat flew out of the cave.",
        "The boy used a bat to hit the ball.",
        "This bag is very light.",
        "We need a match to light the candle."
    ],
    "Ambiguous word": [
        "bank", "bat", "bat", "light", "match"
    ],
    "Intended meaning": [
        "A financial institution",
        "A flying mammal",
        "A sports instrument",
        "Not heavy",
        "A small stick used to create fire"
    ],
    "Contextual clue": [
        "Deposit money indicates a financial institution.",
        "Flying out of a cave indicates an animal.",
        "Hitting a ball indicates sports equipment.",
        "The sentence describes weight.",
        "Lighting a candle indicates fire-making material."
    ]
})

display(ambiguity_examples)


,Sentence,Ambiguous word,Intended meaning,Contextual clue
0,I went to the bank to deposit money.,bank,A financial institution,Deposit money indicates a financial institution.
1,The bat flew out of the cave.,bat,A flying mammal,Flying out of a cave indicates an animal.
2,The boy used a bat to hit the ball.,bat,A sports instrument,Hitting a ball indicates sports equipment.
3,This bag is very light.,light,Not heavy,The sentence describes weight.
4,We need a match to light the candle.,match,A small stick used to create fire,Lighting a candle indicates fire-making material.


In [71]:
# Q4(c): Create at least five coreference examples and identify antecedents.
coreference_examples = pd.DataFrame({
    "Sentence": [
        "Riya submitted her assignment because she had finished it.",
        "The dog chased the cat until it climbed the tree.",
        "Aman bought a laptop because he needed it for college.",
        "The teacher spoke to the student after she arrived.",
        "The car stopped because its engine overheated."
    ],
    "Coreference expression": [
        "her, she, it",
        "it",
        "he, it",
        "she",
        "its"
    ],
    "Antecedent": [
        "Riya; Riya; assignment",
        "cat",
        "Aman; laptop",
        "teacher",
        "car"
    ],
    "Explanation": [
        "Pronouns refer back to Riya and the assignment.",
        "The pronoun 'it' refers to the cat in this context.",
        "The pronouns refer to Aman and the laptop.",
        "The pronoun 'she' refers to the teacher.",
        "The possessive pronoun refers to the car."
    ]
})

display(coreference_examples)


,Sentence,Coreference expression,Antecedent,Explanation
0,Riya submitted her assignment because she had ...,"her, she, it",Riya; Riya; assignment,Pronouns refer back to Riya and the assignment.
1,The dog chased the cat until it climbed the tree.,it,cat,The pronoun 'it' refers to the cat in this con...
2,Aman bought a laptop because he needed it for ...,"he, it",Aman; laptop,The pronouns refer to Aman and the laptop.
3,The teacher spoke to the student after she arr...,she,teacher,The pronoun 'she' refers to the teacher.
4,The car stopped because its engine overheated.,its,car,The possessive pronoun refers to the car.


In [72]:
# Q4(d): Load the sci.space and comp.graphics categories.
categories_q4 = ["sci.space", "comp.graphics"]

newsgroups_q4 = fetch_20newsgroups(
    subset="train",
    categories=categories_q4,
    remove=("headers", "footers", "quotes")
)

print("Number of documents:", len(newsgroups_q4.data))
print("Categories:", newsgroups_q4.target_names)


Number of documents: 1177
Categories: ['comp.graphics', 'sci.space']


In [73]:
# Q4(e): Extract and display at least 20 frequent domain-specific terms.
vectorizer_q4 = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    min_df=2
)

document_term_matrix = vectorizer_q4.fit_transform(newsgroups_q4.data)
term_frequencies = np.asarray(
    document_term_matrix.sum(axis=0)
).ravel()

terms = vectorizer_q4.get_feature_names_out()

domain_terms_q4 = (
    pd.DataFrame({
        "Term": terms,
        "Frequency": term_frequencies
    })
    .sort_values("Frequency", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

display(domain_terms_q4)


,Term,Frequency
0,space,1052
1,image,516
2,data,435
3,edu,429
4,graphics,414
5,nasa,414
6,like,389
7,use,359
8,program,355
9,time,312
